# 08. The mandate: the constraints, the copied optimiser and one month worked through

**What this notebook does.** It sets out the four sets of constraints the study runs, written into `src/slt/constants.py` before any path ran. It checks that the optimiser is the source repository's, unchanged, and works one month of the extraction through it under each of the four sets.

**Why it comes here.** Notebook 07 built the month's matrices of variances and covariances. This notebook shows what the optimiser does with one of them before the script runs it 7,344 times, 24 paths over 306 months. Every rule that decides a weight is in this notebook or in the constants, and the script adds nothing to them.

**Terms used in this notebook.**

- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe weighted by market capitalisation at the start of the month.
- *active weight*: a stock's weight in the portfolio minus its weight in the benchmark. The *active return* is the portfolio's return minus the benchmark's.
- *covariance matrix*: the table of the variances of the members' monthly returns on its diagonal and their covariances off it. Notebook 07 built four for each month, each from a *covariance estimator*: a rule that builds the matrix from three years of daily returns.
- *tracking error*: the standard deviation of the active return, here per year. The *forecast tracking error* is the one the covariance matrix implies for the month's active weights, the square root of 12 times a′Σa, with a the active weights and Σ the month's covariance matrix.
- *optimiser*: the program that chooses the weights with the smallest forecast tracking error under the constraints. The problem is a convex quadratic program, with one minimum, solved by the Clarabel solver through cvxpy.
- *basis point*: a hundredth of a percentage point.
- *turnover*: here one-way turnover, half the sum of the absolute changes in weight, the share of the portfolio sold and bought.
- *reference portfolio*: last month's portfolio after last month's returns, with the stocks that left the universe sold, the stocks that entered bought at their benchmark weight, and the others rescaled to sum to one. *Forced turnover* is the turnover from last month's portfolio to the reference.
- *turnover cap*: the limit on the turnover from the reference portfolio to the new weights, 2 per cent a month on the main paths.
- *tilt*: every stock in GICS sector 10 (Energy) or 55 (Utilities) held at no more than half its benchmark weight.
- *exclusion*: weight zero for GICS sub-industries 30203010 (Tobacco) and 20101010 (Aerospace and Defense). A sub-industry is wider than a list of controversial weapons.
- *beta*: the slope of a stock's return on the benchmark's, from the month's covariance matrix. *Beta neutrality* holds the portfolio's beta equal to the benchmark's, one.
- *constraint set*: the constraints of one run, cumulative in four sets. *C0*: long-only, no negative weight. *C1* adds the bound per stock, which holds each stock within 50 basis points of its benchmark weight, the bound per company, which holds the sum over a company's share classes within the same 50 basis points, and the sector band, which holds each sector within 2 percentage points of its benchmark weight. *C2* adds the turnover cap. *C3*, the *mandate*, adds the tilt, the exclusions and beta neutrality. In a set with the tilt the sector band covers the other sectors only, because half of Energy's weight is above 2 percentage points in 268 of the 306 months, and no portfolio could meet both.
- *binding*: a constraint is binding when the solution sits on its limit, within one millionth.

**What I expect to see.**

- Under C0, C1 and C2 the portfolio is the benchmark, with a forecast tracking error of zero to within the solver's precision: nothing in those sets pushes it away. The sets are run all the same, because the design names them, and reported by year.
- Under C3 the tilt and the exclusions push the portfolio off the benchmark, and the optimiser spends the bounds on the stocks that best hedge what the tilt removed. A forecast tracking error inside the range E6 sets for the realised one, 50 to 150 basis points a year, is expected. E6 itself is judged in notebook 09.
- No company whose share classes differ in sector or in the bound's scope: the count is printed and expected to be zero.

Running time: about three minutes, most of it reading the extraction.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
import hashlib

import numpy as np

from slt import evaluation as V
from slt import mandate as M
from slt.source import optimiser as O

## 1. The copied optimiser

The optimiser is the source repository's `optimiser.py`, copied into `src/slt/source/` with its first line naming the source file and commit and every other line unchanged. The cell recomputes the SHA-256 of each copy without its first line and compares it with the source file's. The study's own module, `src/slt/mandate.py`, builds the same problem from the copy's own pieces and adds the constraints the copy does not take, all of them sums over groups of weights: the bound per company and the sector band. Its tests in `tests/test_mandate.py` solve the source's own constraint sets on made-up months with both and require the same weights within one millionth, a month in which the cap has to be raised included.

In [ ]:
for name, (source, digest) in sorted(C.COPIED_FILES.items()):
    path = ROOT / C.COPIED_INTO / name
    if not path.exists():
        print(f"{name}: copied in milestone {C.COPIED_FOR_MILESTONE[name]}")
        continue
    body = path.read_bytes().split(b"\n", 1)[1]
    ok = hashlib.sha256(body).hexdigest() == digest
    print(f"{name}: {'the source file unchanged' if ok else 'DIFFERS from the source file'} ({digest[:12]})")

## 2. One month worked through

The month is July 2008, when Energy was close to its largest share of the benchmark in the window, so the tilt moves close to the most weight it ever moves. The month's matrices are notebook 07's, the Ledoit-Wolf estimator's is used, and the reference portfolio is the benchmark itself, as on a path's first month, so the turnover cap is left out here: the script applies it from the second month on.

In [ ]:
t0 = time.time()
inp = V.inputs(root)
month = pd.Period("2008-07", "M")
m, r = V.month_view(inp["members"], month)
Sigma = V.month_matrices(month, inp, m)["ledoit_wolf"]
print(f"{time.time() - t0:.0f} s | members with a capitalisation: {len(m.b)} | tilted: {int(m.tilt_mask.sum())}, "
      f"{100 * m.b[m.tilt_mask].sum():.1f} per cent of the benchmark | excluded: {int(m.exclude_mask.sum())}, "
      f"{100 * m.b[m.exclude_mask].sum():.1f} per cent")
worked = {}
for cs, constraints in C.CONSTRAINT_SETS.items():
    cons = tuple(c for c in constraints if c != "turnover_cap")
    res = M.solve(Sigma, m, cons, m.b)
    a = res.weights - m.b
    worked[cs] = {"forecast_te_bp": round(res.forecast_te * 1e4, 4), "held": int((res.weights > C.OPT_TOL).sum()),
                  "names_off_benchmark": int((np.abs(a) > C.OPT_TOL).sum()), "binding": res.binding,
                  "company_scope_mismatches": M.company_scope_mismatches(m, constraints)}
    print(f"{cs}: forecast tracking error {res.forecast_te * 1e4:.2f} basis points a year | stocks held {worked[cs]['held']} | "
          f"off the benchmark {worked[cs]['names_off_benchmark']} | binding {res.binding}")

Under the mandate, the sector active weights of the month, in percentage points (printed here and not committed):

In [ ]:
res = M.solve(Sigma, m, tuple(c for c in C.CONSTRAINT_SETS["C3"] if c != "turnover_cap"), m.b)
a = pd.Series(res.weights - m.b).groupby(m.sector).sum() * 100
beta = O.benchmark_betas(Sigma, m.b) @ res.weights
print(a.round(2).to_string())
print(f"portfolio beta to the benchmark: {beta:.6f}")

## 3. What is committed

One file goes into `results/20260928/`: the worked month's figures under each set, the forecast tracking error, the counts of stocks held and off the benchmark, the binding counts and the company count. No weight and no name.

In [ ]:
record = {"written_by": "notebooks/08_mandate.ipynb", "manifest_sha256": C.MANIFEST_SHA256, "month": str(month),
          "estimator": "ledoit_wolf", "reference": "the benchmark, as on a path's first month; no turnover cap",
          "members": int(len(m.b)), "tilted": int(m.tilt_mask.sum()), "excluded": int(m.exclude_mask.sum()), "sets": worked}
print("written:", X.write_results("mandate_worked_month.json", record=record).relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: that the optimiser is the source's, that the constraint sets do what the constants say on a month of the extraction, and how far the mandate moves the portfolio from the benchmark in one month.

What could still be wrong. One month says nothing about the others: a constraint that does not bind in July 2008 may bind in another month, and the script records every binding count. The worked month starts from the benchmark, so it shows no turnover cap at work. And the forecast tracking error rests on the covariance matrix; whether it is right is the bias statistic of notebook 09.